In [2]:

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("PEDALI-Partie2-RDD")
    .config("spark.driver.host", "pedali-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "512m")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "2")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark :", spark.version)
print("Master :", sc.master)
print("AppName :", sc.appName)

print("=== TEST DU CLUSTER ===")
print("Master :", sc.master)
print("Application :", sc.appName)

resultat = sc.parallelize([10, 20, 30, 40], 2).sum()

print("Resultat :", resultat)



Spark : 3.5.3
Master : spark://spark-master:7077
AppName : PEDALI-Partie2-RDD
=== TEST DU CLUSTER ===
Master : spark://spark-master:7077
Application : PEDALI-Partie2-RDD


[Stage 0:>                                                          (0 + 2) / 2]

Resultat : 100


In [3]:

import json
import time

print("=== CHARGEMENT DES DONNEES PEDALI ===")

debut = time.perf_counter()

chemin = "file:///workspace/data/bronze/trips_history.jsonl"

rdd_brut = sc.textFile(chemin, minPartitions=2)

rdd_trajets = rdd_brut.map(json.loads).cache()

nombre = rdd_trajets.count()

duree = time.perf_counter() - debut

print("Evenements :", nombre)
print("Partitions :", rdd_trajets.getNumPartitions())
print("Temps de chargement :", round(duree, 2), "secondes")
print("Premier evenement :", rdd_trajets.first())

assert nombre == 50000
print("VALIDATION : DONNEES CHARGEES")


=== CHARGEMENT DES DONNEES PEDALI ===


Evenements : 50000
Partitions : 2
Temps de chargement : 1.53 secondes
Premier evenement : {'trip_id': 'T-00000001', 'user_id': 'U-09784', 'vehicle_id': 'V-01989', 'vehicle_type': 'BIKE', 'start_station': 'S-RAB-001', 'end_station': None, 'city': 'Rabat', 'status': 'RESERVED', 'duration_s': None, 'distance_km': None, 'battery_pct': None, 'price_mad': None, 'event_time': '2026-11-15T00:00:00Z'}
VALIDATION : DONNEES CHARGEES


In [4]:

print("=== REPARTITION DES STATUTS ===")

statuts = (
    rdd_trajets
    .map(lambda t: (t.get("status"), 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

for statut, total in sorted(statuts):
    print(statut, ":", total)

print("\n=== EXEMPLE DE TRAJET ENDED ===")

exemples = rdd_trajets.filter(
    lambda t: t.get("status") == "ENDED"
).take(1)

if exemples:
    print(json.dumps(exemples[0], indent=2))
else:
    print("Aucun trajet termine")


=== REPARTITION DES STATUTS ===
CANCELLED : 3453
ENDED : 30228
FAULT_REPORTED : 1507
RESERVED : 7425
UNLOCKED : 7387

=== EXEMPLE DE TRAJET ENDED ===
{
  "trip_id": "T-00000005",
  "user_id": "U-03899",
  "vehicle_id": "V-01206",
  "vehicle_type": "BIKE",
  "start_station": "S-MAR-023",
  "end_station": "S-MAR-015",
  "city": "Marrakech",
  "status": "ENDED",
  "duration_s": 719,
  "distance_km": 0.67,
  "battery_pct": null,
  "price_mad": 10.29,
  "event_time": "2026-11-15T00:00:08Z"
}


In [5]:

rdd_ended = rdd_trajets.filter(
    lambda t: t.get("status") == "ENDED"
).cache()

print("Trajets termines :", rdd_ended.count())


Trajets termines : 30228


In [6]:

ca_par_ville = (
    rdd_ended
    .filter(lambda t: t.get("price_mad") is not None)
    .map(lambda t: (t["city"], float(t["price_mad"])))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
    .collect()
)

print("=== CHIFFRE D'AFFAIRES PAR VILLE ===")

for ville, ca in ca_par_ville:
    print(f"{ville:15s} : {ca:,.2f} MAD")

print(f"\nCA TOTAL : {sum(ca for _, ca in ca_par_ville):,.2f} MAD")


=== CHIFFRE D'AFFAIRES PAR VILLE ===
Casablanca      : 174,698.09 MAD
Marrakech       : 173,034.32 MAD
Tanger          : 172,358.22 MAD
Rabat           : 172,312.90 MAD

CA TOTAL : 692,403.53 MAD


In [7]:

durees = (
    rdd_ended
    .filter(lambda t: t.get("duration_s") is not None)
    .map(
        lambda t: (
            (t["city"], t["vehicle_type"]),
            (float(t["duration_s"]), 1)
        )
    )
    .reduceByKey(
        lambda a, b: (a[0] + b[0], a[1] + b[1])
    )
    .mapValues(lambda x: round(x[0] / x[1], 2))
    .sortByKey()
    .collect()
)

print("=== DUREE MOYENNE DES TRAJETS ===")

for (ville, vehicule), secondes in durees:
    print(
        f"{ville:15s} | {vehicule:10s} | "
        f"{secondes / 60:.2f} minutes"
    )


=== DUREE MOYENNE DES TRAJETS ===
Casablanca      | BIKE       | 31.77 minutes
Casablanca      | EBIKE      | 31.70 minutes
Casablanca      | SCOOTER    | 31.44 minutes
Marrakech       | BIKE       | 31.33 minutes
Marrakech       | EBIKE      | 31.42 minutes
Marrakech       | SCOOTER    | 31.59 minutes
Rabat           | BIKE       | 31.34 minutes
Rabat           | EBIKE      | 31.15 minutes
Rabat           | SCOOTER    | 31.14 minutes
Tanger          | BIKE       | 31.23 minutes
Tanger          | EBIKE      | 31.68 minutes
Tanger          | SCOOTER    | 30.94 minutes


In [8]:

top_stations = (
    rdd_ended
    .filter(lambda t: t.get("start_station") is not None)
    .map(lambda t: (t["start_station"], 1))
    .reduceByKey(lambda a, b: a + b)
    .takeOrdered(5, key=lambda x: -x[1])
)

print("=== TOP 5 DES STATIONS DE DEPART ===")

for rang, (station, nombre) in enumerate(top_stations, 1):
    print(f"{rang}. {station} : {nombre} trajets")


=== TOP 5 DES STATIONS DE DEPART ===
1. S-CAS-003 : 900 trajets
2. S-TAN-004 : 864 trajets
3. S-MAR-004 : 857 trajets
4. S-TAN-003 : 856 trajets
5. S-RAB-004 : 855 trajets


In [10]:

print("=== LINEAGE DU RDD ===")

rdd_ca = (
    rdd_ended
    .filter(lambda t: t.get("price_mad") is not None)
    .map(lambda t: (t["city"], float(t["price_mad"])))
    .reduceByKey(lambda a, b: a + b)
)

print(rdd_ca.toDebugString().decode("utf-8"))

print("\n=== EXECUTION DU JOB ===")

sc.setJobGroup(
    "PEDALI_CA_RDD",
    "PEDALI - Chiffre d'affaires par ville avec reduceByKey"
)

resultats = rdd_ca.collect()

for ville, ca in sorted(resultats):
    print(f"{ville} : {ca:.2f} MAD")

sc.setLocalProperty("spark.jobGroup.id", None)
sc.setLocalProperty("spark.job.description", None)
sc.setLocalProperty("spark.job.interruptOnCancel", None)


=== LINEAGE DU RDD ===
(2) PythonRDD[51] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[50] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[49] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[48] at reduceByKey at /tmp/ipykernel_493/2682061066.py:7 []
    |  PythonRDD[47] at reduceByKey at /tmp/ipykernel_493/2682061066.py:7 []
    |  PythonRDD[13] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 2; MemorySize: 1995.6 KiB; DiskSize: 0.0 B
    |  PythonRDD[4] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 2; MemorySize: 3.0 MiB; DiskSize: 0.0 B
    |  file:///workspace/data/bronze/trips_history.jsonl MapPartitionsRDD[3] at textFile at NativeMethodAccessorImpl.java:0 []
    |  file:///workspace/data/bronze/trips_history.jsonl HadoopRDD[2] at textFile at NativeMethodAccessorImpl.java:0 []

=== EXECUTION DU JOB ===
Casablanca : 174698.09 MAD
Marrakech : 173034.32 MAD
Rabat : 172312.90 MAD
Tanger : 172358.22 MAD


In [11]:

print("Application :", sc.appName)
print("Application ID :", sc.applicationId)
print("Spark UI :", sc.uiWebUrl)


Application : PEDALI-Partie2-RDD
Application ID : app-20261125120217-0000
Spark UI : http://pedali-jupyter:4041


In [ ]:

import json

chemin = "file:///workspace/data/bronze/trips_history.jsonl"

rdd_brut = sc.textFile(chemin, minPartitions=2)

rdd_trajets = rdd_brut.map(json.loads)

print("Nombre d'evenements :", rdd_trajets.count())
print("Premier evenement :", rdd_trajets.first())
print("Nombre de partitions :", rdd_trajets.getNumPartitions())


In [ ]:

print("=== CHAMPS DISPONIBLES ===")
print(list(rdd_trajets.first().keys()))

print("\n=== REPARTITION PAR STATUT ===")
statuts = (
    rdd_trajets
    .map(lambda trajet: (trajet.get("status"), 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

for statut, total in sorted(statuts, key=lambda x: str(x[0])):
    print(statut, ":", total)

print("\n=== EXEMPLE DE TRAJET TERMINE ===")
exemple = rdd_trajets.filter(
    lambda trajet: trajet.get("status") == "ENDED"
).take(1)

print(exemple[0] if exemple else "Aucun trajet ENDED")


In [ ]:

docker logs pedali-spark-worker-1 --tail 30

docker logs pedali-spark-worker-2 --tail 30
